# Classifier free guidance - edges study

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import mediapy
import cv2
from torch.nn.functional import interpolate

sys.path.insert(0, '../../..')             # repo root

from dreamerv4uwm.models.utils import load_tokenizer, load_denoiser
from dreamerv4uwm.datasets import G1ChunkDataset, ShardedHDF5Dataset
from dreamerv4uwm.sampling_new import make_is_horizon, _quantize_tau_to_idx
from dreamerv4uwm.inference.hybrid_chunk import HybridChunkSampler

torch.manual_seed(0)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
resolution = (256, 256)
print('device:', device)

In [ ]:
from hydra import initialize_config_dir, compose
from dreamerv4uwm.models.utils import load_tokenizer, load_denoiser
from dreamerv4uwm.datasets import ShardedHDF5Dataset
from dreamerv4uwm.planning import rollout_new as R
# from dreamerv4uwm.planning import rollout_stoch as RS

CFG_DIR  = '/home/mim-server/projects/felix/dreamerV4-UWM/scripts/config'
DYN_CKPT = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/blockcausal/pushT-post-train/97500.pt'
TOK_CKPT = '/home/mim-server/projects/rooholla/dreamerV4-UWM/checkpoints/tokenizer/pushT.pt'
DATA_PATH = '/home/mim-server/datasets/pushT/h5/play'

with initialize_config_dir(version_base=None, config_dir=CFG_DIR):
    cfg = compose(config_name='dynamics/pushT-large', overrides=['denoiser.horizon_aware=false'])
cfg.dynamics_ckpt, cfg.tokenizer_ckpt = DYN_CKPT, TOK_CKPT
denoiser  = load_denoiser(cfg, device, max_num_forward_steps=300).eval().to(device)
tokenizer = load_tokenizer(cfg, device, max_num_forward_steps=300).eval().to(device)
print('n_actions:', cfg.denoiser.n_actions)

In [ ]:
dataset = ShardedHDF5Dataset(data_dir=DATA_PATH, window_size=64, stride=1, split='train',
                             train_fraction=0.9, split_seed=123, shuffle_windows=False)
print('dataset size:', len(dataset))

## Display helpers

In [ ]:
@torch.no_grad()
def decode(lat):
    """Latents (B,T,N,D) -> video (B,T,3,H,W) float[0,1] (bf16 autocast)."""
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        v = tokenizer.decode(lat.to(device))
    return v.float().clamp(0, 1)

def frame_rgb(lat_1frame):
    """(1,1,N,D) latent -> (H,W,3) uint8 RGB."""
    v = decode(lat_1frame)[0, 0]
    return (v.permute(1, 2, 0).clamp(0, 1) * 255).to(torch.uint8).cpu().numpy()

def _strip(video_tchw, n_frames=8, border=2):
    """(T,C,H,W) -> one (H,W,C) uint8 filmstrip, frames evenly spaced, white separators."""
    T = video_tchw.shape[0]
    idx = np.linspace(0, T - 1, min(n_frames, T), dtype=int)
    fr = (video_tchw[idx].cpu().permute(0, 2, 3, 1).float().numpy() * 255).clip(0, 255).astype(np.uint8)
    H, W, C = fr.shape[1:]
    sep = np.full((H, border, C), 255, np.uint8)
    parts = []
    for i, f in enumerate(fr):
        if i:
            parts.append(sep)
        parts.append(f)
    return np.concatenate(parts, axis=1)

def plotSnapshots(video, n_frames=8, figsize=None):
    """Show a single clip as one filmstrip row."""
    strip = _strip(video, n_frames)
    figsize = figsize or (n_frames * 2, 2.2)
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(strip); ax.axis('off'); plt.tight_layout(); plt.show()

def plotComparison(named_videos, n_frames=8, title=None):
    """Stack labeled (T,C,H,W) clips as filmstrip rows (one row per clip)."""
    rows = [(lab, _strip(v, n_frames)) for lab, v in named_videos]
    fig, axes = plt.subplots(len(rows), 1, figsize=(n_frames * 2, 2.0 * len(rows)))
    if len(rows) == 1:
        axes = [axes]
    for ax, (lab, img) in zip(axes, rows):
        ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylabel(lab, rotation=0, ha='right', va='center', fontsize=10)
    if title:
        axes[0].set_title(title, fontsize=12)
    plt.tight_layout(); plt.show()

def plotVideo(video, fps=10):
    """Play a (T,C,H,W) float[0,1] clip inline, with frame numbers burned in."""
    arr = (video.cpu().permute(0, 2, 3, 1).to(torch.float32).numpy() * 255).clip(0, 255).astype(np.uint8)
    for i in range(arr.shape[0]):
        cv2.putText(arr[i], f'Frame {i}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    mediapy.show_video(arr, fps=fps)

def get_window(idx):
    batch = dataset[idx]
    imgs = interpolate(batch['image'], resolution).to(device)[None]
    actions = batch['action'][:, :cfg.denoiser.n_actions][None].to(device)
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        latents = tokenizer.encode(imgs).float()
    return imgs, actions, latents

def plotTerminals(vid, title=None, ncol=None):
    """Grid of the last frame of each sibling video (B,T,3,h,w)."""
    B_ = vid.shape[0]
    ncol = ncol or B_
    nrow = math.ceil(B_ / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.2 * ncol, 2.4 * nrow), squeeze=False)
    for i, ax in enumerate(axes.flat):
        if i < B_:
            ax.imshow(vid[i, -1].permute(1, 2, 0).cpu().numpy())
            ax.set_title(f'b={i}', fontsize=9)
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout(); plt.show()

def plotSiblingVideos(vid, fps=8, columns=3):
    """Tile the B sibling clips into one inline video grid."""
    arrs = {}
    for b in range(vid.shape[0]):
        a_ = (vid[b].cpu().permute(0, 2, 3, 1).float().numpy() * 255).clip(0, 255).astype(np.uint8)
        arrs[f'b={b}'] = a_
    mediapy.show_videos(arrs, fps=fps, columns=columns, height=200)

def plotActionPaths(a, ca=None, title=None):
    """Overlay the B action sequences (B,H,2); grey = context actions (1,Tc,2)."""
    a_np = a.float().cpu().numpy()
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

    if ca is not None:
        c = ca[0].float().cpu().numpy()
        ax[0].plot(c[:, 0], c[:, 1], '-o', c='0.6', ms=3, lw=1.5, label='context (real)')
        ax[0].scatter(c[-1, 0], c[-1, 1], c='k', s=60, zorder=5, label='decision point')
    for b in range(a_np.shape[0]):
        ax[0].plot(a_np[b, :, 0], a_np[b, :, 1], '-o', ms=3, alpha=.85, label=f'b={b}')
        ax[1].plot(a_np[b, :, 0], alpha=.85)
        ax[2].plot(a_np[b, :, 1], alpha=.85)
    ax[0].set(xlabel='action dim 0', ylabel='action dim 1', title='action paths in 2-D')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3); ax[0].set_aspect('equal', 'datalim')
    ax[1].set(xlabel='frame', ylabel='action dim 0', title='dim 0 vs time'); ax[1].grid(alpha=.3)
    ax[2].set(xlabel='frame', ylabel='action dim 1', title='dim 1 vs time'); ax[2].grid(alpha=.3)
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout(); plt.show()

## The context (T_c=1)

In [ ]:
WINDOW_SEED = 1234
T0     = 50        # decision at frame T0 + T_CTX
T_CTX  = 1         # context frames
B      = 6         # siblings drawn from this one context
H      = 16        # edge horizon (frames per rollout)
K      = 8         # Euler steps

imgs, actions, latents = get_window(WINDOW_SEED)
cz = latents[:, T0:T0 + T_CTX].clone()      # (1,Tc,N,D)
ca = actions[:, T0:T0 + T_CTX].clone()      # (1,Tc,n_act)

plotComparison([('context', decode(cz)[0])], n_frames=T_CTX, title='the context all siblings share')
plt.figure(figsize=(3.4, 3.4))
plt.imshow(frame_rgb(cz[:, -1:])); plt.axis('off'); plt.title('decision frame'); plt.show()

## Functions

In [ ]:
from dreamerv4uwm.planning.rollout_new import _dims, _resolve_device, _expand_ctx, _scaled_noise, policy, transition
from typing import Optional, Tuple

def _advance_ctx_autoreg(cz, ca, z1, a1, max_len=None):
    """Append the new state and action to the context, optionally truncating to max_len."""
    cz = z1 if cz is None else torch.cat([cz, z1], dim=1)
    ca = a1 if ca is None else torch.cat([ca, a1], dim=1)
    if max_len is not None:
        cz, ca = cz[:, -max_len:], ca[:, -max_len:]
    return cz, ca

def const_policy(denoiser, del_X, del_Y):
    def policy_fn(ctx_z, ctx_a, H, B=1, K=12, *, ctx_noise=0.0, ctx_noise_honest=True,
                  action_temp=1.0, action_prior="normal", state_prior="normal",
                  dtype=None, device=None, generator=None):
        _, N_lat, D_lat, n_act = _dims(denoiser)
        device = _resolve_device(denoiser, ctx_z, device=device)
        cz, ca = _expand_ctx(ctx_z, ctx_a, B)
        if cz is not None:
            cz, ca = cz.contiguous(), ca.contiguous()
        a_out = torch.empty(B, H, n_act, device=device)
        for h in range(H):
            a1 = torch.tensor(np.array([[[del_X, del_Y]]]), device=device).repeat(B, 1, 1)
            a_out[:, h] = a1[:, 0]
            z1 = transition(denoiser, cz, ca, a1, K=K, ctx_noise=ctx_noise,
                            ctx_noise_honest=ctx_noise_honest, state_prior=state_prior,
                            dtype=dtype, device=device, generator=generator)
            cz = z1 if cz is None else torch.cat([cz, z1], dim=1)
            ca = a1 if ca is None else torch.cat([ca, a1], dim=1)
        return a_out
    return policy_fn
    

@torch.no_grad()
def autoregressive_new(
    denoiser,
    ctx_z: Optional[torch.Tensor],  # (1|B, Tc, N_lat, D_lat), or None
    ctx_a: Optional[torch.Tensor],  # (1|B, Tc, n_act),        or None
    H: int,
    B: int = 1,
    K: int = 12,
    *,
    ctx_len: Optional[int] = None,
    display: bool = False,
    determ: bool = False,
    del_X: float = -0.1006,
    del_Y: float = 0.1000,
    ctx_noise: float = 0.0,
    ctx_noise_honest: bool = True,
    action_temp: float = 1.0,
    action_prior: str = "normal",
    state_prior: str = "normal",
    action_noise: float = 0.0,
    action_noise_dist: str = "normal",
    dtype: Optional[torch.dtype] = torch.bfloat16,
    device: Optional[torch.device] = None,
    generator: Optional[torch.Generator] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:

    _, N_lat, D_lat, n_act = _dims(denoiser)
    device = _resolve_device(denoiser, ctx_z, device=device)
    cz, ca = _expand_ctx(ctx_z, ctx_a, B)
    if cz is not None:
        cz, ca = cz.contiguous(), ca.contiguous()
    z_out = torch.empty(B, H, N_lat, D_lat, device=device)
    a_out = torch.empty(B, H, n_act, device=device)
    cst_policy = const_policy(denoiser, del_X=del_X, del_Y=del_Y)
    for h in range(H):
        # 1. one action from the policy (horizon 1; the policy's own future state is noise)
        if determ:
            a1 = cst_policy(cz, ca, 1, B=B, K=K, ctx_noise=ctx_noise,
                            ctx_noise_honest=ctx_noise_honest, action_temp=action_temp,
                            action_prior=action_prior, state_prior=state_prior,
                            dtype=dtype, device=device, generator=generator)   # (B,1,n_act)
        else:
            a1 = policy(denoiser, cz, ca, 1, B=B, K=K, ctx_noise=ctx_noise,
                        ctx_noise_honest=ctx_noise_honest, action_temp=action_temp,
                        action_prior=action_prior, state_prior=state_prior,
                        dtype=dtype, device=device, generator=generator)   # (B,1,n_act)
        if display:
            print(f'frame {h}: action {a1[:, 0].cpu().numpy()}')
        # 2. optional additive exploration noise on the sampled action
        if action_noise and action_noise > 0.0:
            a1 = a1 + _scaled_noise((B, 1, n_act), scale=action_noise, dist=action_noise_dist,
                                    device=device, generator=generator)
        # 3. world-model step to the next state given that action (horizon 1)
        z1 = transition(denoiser, cz, ca, a1, K=K, ctx_noise=ctx_noise,
                        ctx_noise_honest=ctx_noise_honest, state_prior=state_prior,
                        dtype=dtype, device=device, generator=generator)  # (B,1,N,D)
        z_out[:, h], a_out[:, h] = z1[:, 0], a1[:, 0]
        if display: # decode the last B generated latents 
            v = decode(z_out[:, :h + 1])
            plotComparison([(f'b={b}', v[b]) for b in range(B)], n_frames=h + 1,
                           title=f'{B} autoregressive rollouts from the given context')
        # 4. append the realized (state, action) and advance the context
        #    (from nothing, on the unconditional first step)
        cz, ca = _advance_ctx_autoreg(cz, ca, z1, a1, max_len=ctx_len)
    return z_out, a_out

## Classifier free guidance

In [ ]:
@torch.no_grad()
def getVelField(denoiser, z, tau, context=None, context_tau=0.9, context_at_end=False):
    """Query the denoiser once at cleanness ``tau``; return (x_pred, velocity)
    for the horizon frames ``z``.

    Args:
        denoiser: DenoiserWrapper.
        z: horizon latents being integrated (1, T_hor, N_lat, D_lat).
        tau: current cleanness in [0, 1) (0 = pure noise).
        context: optional clean latents (1, T_ctx, N_lat, D_lat) to condition on.
        context_tau: cleanness pinned on the context frames (~1 = clean).
        context_at_end: append context after the horizon instead of before
            (use a *future* anchor, i.e. goal-conditioning).

    Returns:
        (x, v): x-prediction and velocity, both sliced back to the horizon
        frames only — shapes (1, T_hor, N_lat, D_lat).
    """
    d = denoiser.cfg.denoiser
    N, n_act = d.num_noise_levels, d.n_actions
    device = z.device
    # Assemble the attended sequence (horizon +/- context).
    if context is None:
        z_t = z
    elif context_at_end:
        z_t = torch.cat([z, context], dim=1)
    else:
        z_t = torch.cat([context, z], dim=1)
    T = z_t.shape[1]
    is_hor = make_is_horizon(T, all_bidir=True, device=device)         # ignored when horizon_aware=false
    a = torch.randn(1, T, n_act, device=device)
    step_idx = torch.zeros((1, T), dtype=torch.long, device=device)
    obs_sigma = torch.full((1, T), _quantize_tau_to_idx(tau, N), dtype=torch.long, device=device)
    act_sigma = torch.full((1, T), _quantize_tau_to_idx(0.0, N), dtype=torch.long, device=device)
    if context is not None:                                            # pin context frames clean
        Tc = context.shape[1]
        ctx_idx = _quantize_tau_to_idx(context_tau, N)
        if context_at_end:
            obs_sigma[:, -Tc:] = ctx_idx
        else:
            obs_sigma[:, :Tc] = ctx_idx
    z_hat, _, _ = denoiser(
        noisy_act=a, noisy_obs=z_t,
        obs_sigma_idx=obs_sigma, obs_step_idx=step_idx,
        act_sigma_idx=act_sigma, act_step_idx=step_idx, is_horizon=is_hor,
    )
    v = (z_hat - z_t) / (1.0 - tau)
    x = z_hat
    if context is None:
        return x, v
    return (x[:, :-Tc], v[:, :-Tc]) if context_at_end else (x[:, Tc:], v[:, Tc:])

In [ ]:
@torch.no_grad()
def videoSampler(denoiser, num_diffusion_steps, video_length=16):
    """Unconditional generation via flow-matching Euler integration.

    Args:
        denoiser: DenoiserWrapper.
        num_diffusion_steps: number of Euler steps K.
        video_length: number of frames to generate.

    Returns:
        Final x-prediction latents (1, video_length, N_lat, D_lat).
    """
    d = denoiser.cfg.denoiser
    N, N_lat, D_lat, n_act = d.num_noise_levels, d.num_latent_tokens, d.latent_dim, d.n_actions
    device = next(denoiser.parameters()).device
    is_hor = make_is_horizon(video_length, all_bidir=True, device=device)
    z = torch.randn(1, video_length, N_lat, D_lat, device=device)     # prior (tau=0)
    a = torch.randn(1, video_length, n_act, device=device)
    K = int(num_diffusion_steps)
    step_idx = torch.zeros((1, video_length), dtype=torch.long, device=device)
    act_sigma = torch.full((1, video_length), _quantize_tau_to_idx(0.0, N), dtype=torch.long, device=device)
    obs_sigma = torch.zeros((1, video_length), dtype=torch.long, device=device)
    dt = 1.0 / K
    for k in range(K):
        cur = k / K
        obs_sigma[:] = _quantize_tau_to_idx(cur, N)
        z_hat, _, _ = denoiser(
            noisy_act=a, noisy_obs=z,
            obs_sigma_idx=obs_sigma, obs_step_idx=step_idx,
            act_sigma_idx=act_sigma, act_step_idx=step_idx, is_horizon=is_hor,
        )
        z = z + (z_hat - z) / (1.0 - cur) * dt
    return z_hat


z_uncond = videoSampler(denoiser, num_diffusion_steps=16, video_length=8)
plotSnapshots(decode(z_uncond)[0], n_frames=8)